In [4]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [5]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [12]:
def get_single_bit_result(circuit):
    backend = BasicSimulator()
    compiled_circuit = transpile(circuit, backend)
    circuit_sim = backend.run(compiled_circuit, shots=1)
    return int(list(circuit_sim.result().get_counts().keys())[0])

def get_two_bit_result(circuit):
    backend = BasicSimulator()
    compiled_circuit = transpile(circuit, backend)
    circuit_sim = backend.run(compiled_circuit, shots=1)
    result = list(circuit_sim.result().get_counts().keys())[0]
    return (int(result[0]), int(result[1]))

def random_of_2():
    random_circuit = QuantumCircuit(1,1)
    random_circuit.h(0)
    random_circuit.measure(0,0)
    return get_single_bit_result(random_circuit)

def random_of_3():
    U = [ [1/math.sqrt(3),-math.sqrt(2)/math.sqrt(3)],
          [math.sqrt(2)/math.sqrt(3),1/math.sqrt(3) ] ]
    
    random_circuit = QuantumCircuit(1,1)
    random_circuit.unitary(U,0)
    random_circuit.measure(0,0)
    if get_single_bit_result(random_circuit) == 1:
        return random_of_2() + 1
    else:
        return 0

def entangler():
    circuit = QuantumCircuit(2,2)
    circuit.x(0)
    circuit.x(1)
    circuit.h(0)
    circuit.cx(0,1)
    return circuit

def avg(l):
    if len(l)==0:
        return 0
    else:
        return sum(l)/len(l)

DENOM1 = math.sqrt(4 + 2*math.sqrt(2))
DENOM2 = math.sqrt(4 - 2*math.sqrt(2)) 
W_TRASNFORM = [ [ -1/DENOM1 , (1 + math.sqrt(2))/DENOM1 ],
                [  1/DENOM2 , (math.sqrt(2) - 1)/DENOM2 ] ]
V_TRANSFORM = [ [  1/DENOM1 , (1 + math.sqrt(2))/DENOM1 ],
                [ -1/DENOM2 , (math.sqrt(2) - 1)/DENOM2 ] ]
KEY_LENGTH = 16
ATTACKER_THRESHOLD = 2.4

alice_bases = []
bob_bases = []
alice_bits = []
bob_bits = []

for i in range((9*KEY_LENGTH)//2):
    #Alice and Bob are sent entangled qubit
    circuit = entangler()
    
    #Alice chooses operator
    alice_choice = random_of_3()
    alice_bases.append(alice_choice)
    if alice_choice == 0:
        #diagonal
        circuit.h(0)
    elif alice_choice == 1:
        #w basis
        circuit.unitary(W_TRASNFORM, 0)
    else:
        #standard
        pass
    
    #Bob chooses operator
    bob_choice = random_of_3()
    bob_bases.append(bob_choice)
    if bob_choice == 0:
        #w basis
        circuit.unitary(W_TRASNFORM, 1)
    elif bob_choice == 1:
        #standard
        pass
    else:
        #v basis
        circuit.unitary(V_TRANSFORM, 1)
    
    #Alice measures
    circuit.measure(0, 0)
    
    #Bob measures
    circuit.measure(1, 1)

    alice_bit, bob_bit = get_two_bit_result(circuit)
    alice_bits.append(alice_bit)
    bob_bits.append(bob_bit)
    
#Operator choices compared
alice_key = []
bob_key = []
x_w = []
x_v = []
z_w = []
z_v = []
for i,alice_basis in enumerate(alice_bases):
    if (alice_basis == 1 and bob_bases[i] == 0) or (alice_basis == 2 and bob_bases[i] == 1):
        alice_key.append(alice_bits[i])
        bob_key.append(1-bob_bits[i])
    else:
        result = (1-2*alice_bits[i])*(1-2*bob_bits[i])
        if (alice_basis == 0 and bob_bases[i] == 0):
            x_w.append(result)
        elif (alice_basis == 0 and bob_bases[i] == 2):
            x_v.append(result)
        elif (alice_basis == 2 and bob_bases[i] == 0):
            z_w.append(result)
        elif (alice_basis == 2 and bob_bases[i] == 2):
            z_v.append(result)

print(f"alice key: {alice_key}")
print(f"bob key:{bob_key}")

s = abs(avg(x_w) - avg(x_v) + avg(z_w) + avg(z_v))
if s<=ATTACKER_THRESHOLD:
    print(f"!!attacker detected!!, s:{s}")

alice key: [0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1]
bob key:[0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1]
